# Seven-day window evidence review
Read-only reproduction of the supplemental audit summary. Run from this notebook directory. Original data, model weights and scores are unchanged.
Official end time is a last-good-data boundary, not a measured first-grounding timestamp. Zero after-boundary endpoints is not a zero-grounding-rate estimate.

In [1]:
from pathlib import Path
import pandas as pd
source = Path.cwd().parent / "coastal_grounding_20261110T043836Z" / "results"
a = pd.read_parquet(source / "origin_availability.parquet")
assert not a.duplicated(["id", "origin_source_obs", "horizon_days"]).any()
assert (a.target_epoch - a.origin_epoch).eq(7 * 86400).all()
used = a[a.used_by_original]
assert not used.target_epoch.gt(used.official_end_epoch).any()
rows = []
for cohort, group in a.groupby("cohort"):
    g = group[group.used_by_original & group.official_typedeath.eq(1)]
    window = group[group.official_typedeath.eq(1) & group.origin_epoch.lt(group.official_end_epoch) & group.target_epoch.ge(group.official_end_epoch)]
    rows.append({"cohort": cohort, "used_from_code1": len(g),
                 "before": int(g.target_epoch.lt(g.official_end_epoch).sum()),
                 "equal": int(g.target_epoch.eq(g.official_end_epoch).sum()),
                 "after": int(g.target_epoch.gt(g.official_end_epoch).sum()),
                 "missing_related_windows": int(window.missing_endpoint.sum())})
print(pd.DataFrame(rows).to_string(index=False))


    cohort  used_from_code1  before  equal  after  missing_related_windows
     train            38522   38521      1      0                      490
validation             8544    8543      1      0                      167
